# Validação do challenger V2 da IA dos sensores

Compara o Isolation Forest V1 com o Extra Trees V2 no mesmo trecho final. O V2 aprende com perturbações sintéticas; os resultados medem sensibilidade técnica, não acurácia sanitária.

In [ ]:
from pathlib import Path
REPO_DIR = Path('/content/tcc-vigilancia-wbe-iot')
if not REPO_DIR.exists():
    !git clone -q --branch feature/iot-anomaly-ai --single-branch https://github.com/samuelfzst-bit/tcc-vigilancia-wbe-iot.git $REPO_DIR
else:
    !git -C $REPO_DIR fetch -q origin feature/iot-anomaly-ai
    !git -C $REPO_DIR switch -q feature/iot-anomaly-ai
    !git -C $REPO_DIR pull -q --ff-only origin feature/iot-anomaly-ai
%cd /content/tcc-vigilancia-wbe-iot
!python -m pip install -q -r requirements/frontend.txt
print('Ambiente preparado.')

## Ler a telemetria real em modo somente leitura

In [ ]:
from google.colab import userdata
from sqlalchemy import create_engine, text
import pandas as pd
import matplotlib.pyplot as plt
from frontend.data_access import database_url
DB_URL = userdata.get('TCC_DB_URL')
engine = create_engine(database_url({'TCC_DB_URL': DB_URL}), pool_pre_ping=True, connect_args={'connect_timeout': 15})
QUERY = text("""select id, device_id, data_hora, ph_value, turbidity_value, tds_value, temperature_value, status from public.leituras_sensor where data_hora >= timestamptz '2026-01-01' order by data_hora, id""")
with engine.connect() as connection:
    leituras = pd.read_sql(QUERY, connection, parse_dates=['data_hora'])
print('Dimensão:', leituras.shape, '| Período:', leituras.data_hora.min(), 'a', leituras.data_hora.max())

## Ajustar, calibrar e testar sem sobreposição temporal

In [ ]:
from src.sensor_challenger_v2 import temporal_validation_v2, evaluate_perturbation_grid_v2
from src.sensor_validation import temporal_validation, evaluate_perturbation_grid, acceptance_criteria
v1 = temporal_validation(leituras, reference_end='2026-11-04T00:00:00Z', train_fraction=0.8)
v2 = temporal_validation_v2(leituras, reference_end='2026-11-04T00:00:00Z')
assert v2.fit.data_hora.max() < v2.calibration.data_hora.min()
assert v2.calibration.data_hora.max() < v2.test.data_hora.min()
display(pd.Series(v2.summary, name='valor').to_frame())
print('Barreiras temporais confirmadas.')

In [ ]:
grade_v1 = evaluate_perturbation_grid(v1)
grade_v2 = evaluate_perturbation_grid_v2(v2)
criterios_v1 = acceptance_criteria(grade_v1).assign(modelo='V1 Isolation Forest')
criterios_v2 = acceptance_criteria(grade_v2).assign(modelo='V2 Extra Trees')
comparacao = pd.concat([criterios_v1, criterios_v2], ignore_index=True)
display(comparacao.style.format({'observed': '{:.1%}', 'target': '{:.1%}'}))
print('V1:', int(criterios_v1.passed.sum()), '/ 7 | V2:', int(criterios_v2.passed.sum()), '/ 7')

In [ ]:
single = grade_v2.query("scenario == 'single_sensor'")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for sensor, grupo in single.groupby('sensor'):
    axes[0].plot(grupo.magnitude, grupo.detection_rate, marker='o', label=sensor)
axes[0].axhline(.5, color='red', linestyle='--', label='meta forte')
axes[0].set(title='V2 por sensor e intensidade', xlabel='Magnitude', ylabel='Taxa sinalizada', ylim=(0, 1.05))
axes[0].legend()
multi = grade_v2.query("scenario == 'multivariate'")
axes[1].plot(multi.magnitude, multi.detection_rate, marker='o')
axes[1].axhline(.7, color='orange', linestyle='--', label='meta moderada')
axes[1].axhline(.9, color='red', linestyle='--', label='meta forte')
axes[1].set(title='V2 multivariado', xlabel='Escala', ylabel='Taxa sinalizada', ylim=(0, 1.05))
axes[1].legend()
for ax in axes: ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## Exportar evidências — passar 7/7 não promove automaticamente o V2; ainda falta o ensaio físico.

In [ ]:
import json
OUT = Path('/content/resultados_challenger_v2')
OUT.mkdir(exist_ok=True)
grade_v2.to_csv(OUT / 'grade_perturbacoes_v2.csv', index=False)
comparacao.to_csv(OUT / 'comparacao_criterios_v1_v2.csv', index=False)
resumo = {k: (v.isoformat() if hasattr(v, 'isoformat') else v) for k, v in v2.summary.items()}
(OUT / 'resumo_temporal_v2.json').write_text(json.dumps(resumo, indent=2), encoding='utf-8')
!cd /content && zip -qr resultados_challenger_v2.zip resultados_challenger_v2
print('Gerado: /content/resultados_challenger_v2.zip')